In [ ]:
# Medical Data Warehouse

## Exploration des données XML PubMed

### Objectif
Analyser les documents XML médicaux afin de comprendre
leur structure, identifier les informations disponibles
et préparer les données pour la construction d'un
entrepôt de données.

### Dataset
- Source : PubMed
- Format : XML
- Nombre de fichiers : 4036

In [1]:
import os
import pandas as pd
from lxml import etree
import matplotlib.pyplot as plt
from pathlib import Path

print("Bibliothèques importées avec succès !")

Bibliothèques importées avec succès !


In [2]:
# Chercher le dossier raw
possible_paths = [
    Path("data/raw"),
    Path("../data/raw"),
    Path.cwd() / "data" / "raw",
    Path.cwd().parent / "data" / "raw"
]

DATA_DIR = next(
    (p.resolve() for p in possible_paths if p.exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Le dossier data/raw est introuvable."
    )

# Récupérer les fichiers XML
xml_files = sorted(DATA_DIR.glob("*.xml"))

print("Dossier des données :", DATA_DIR)
print("Nombre de fichiers XML :", len(xml_files))
print("Trois premiers fichiers :")

for file in xml_files[:3]:
    print(file.name)

Dossier des données : C:\Users\Ghada\Desktop\medical-data-warehouse\data\raw
Nombre de fichiers XML : 4036
Trois premiers fichiers :
1000.xml
1001.xml
1002.xml


In [3]:
# Sélectionner le premier fichier
first_file = xml_files[0]

print("Fichier analysé :", first_file.name)

# Lire le document XML
tree = etree.parse(str(first_file))

# Récupérer la racine
root = tree.getroot()

print("\nBalise racine :", root.tag)

print("\nPremières balises :")

for element in list(root)[:10]:
    print("-", element.tag)

Fichier analysé : 1000.xml

Balise racine : PubmedArticleSet

Premières balises :
- PubmedArticle


In [4]:
# Chercher un article PubMed
articles = root.xpath(
    "//*[local-name()='PubmedArticle']"
)

if articles:
    article = articles[0]

    print("Article trouvé !")

    # Afficher les balises et leurs valeurs
    for element in article.iter():
        if element.text and element.text.strip():
            print(
                element.tag.split("}")[-1],
                ":",
                element.text.strip()[:150]
            )
else:
    print("Aucun article trouvé dans ce document.")

Article trouvé !
PMID : 26613881
Year : 2015
Month : 11
Day : 28
ISSN : 1557-9786
Volume : 24
Issue : 1
Year : 2016
Month : Feb
Title : Magnetic resonance imaging clinics of North America
ISOAbbreviation : Magn Reson Imaging Clin N Am
ArticleTitle : Role of Multiparametric MR Imaging in Malignancies of the Urogenital Tract.
MedlinePgn : 187-204
ELocationID : 10.1016/j.mric.2015.08.009
ELocationID : S1064-9689(15)00107-5
AbstractText : Multiparametric MR imaging (mpMRI) combine different sequences that, properly tailored, can provide qualitative and quantitative information about the
CopyrightInformation : Copyright © 2016 Elsevier Inc. All rights reserved.
LastName : de Leon
ForeName : Alberto Diaz
Initials : AD
Affiliation : Department of Radiology, University of Texas Southwestern Medical Center, 2201 Inwood Road, 2nd Floor, Suite 202, Dallas, TX, USA.
LastName : Costa
ForeName : Daniel
Initials : D
Affiliation : Department of Radiology, University of Texas Southwestern Medical Cente

In [5]:
from collections import Counter

tag_counter = Counter()
file_errors = []

for file in xml_files:

    try:
        tree = etree.parse(str(file))
        root = tree.getroot()

        for element in root.iter():

            # Retirer un éventuel namespace XML
            tag = element.tag.split("}")[-1]

            tag_counter[tag] += 1

    except Exception as e:

        file_errors.append({
            "file": file.name,
            "error": str(e)
        })

print("Nombre de fichiers analysés :", len(xml_files))
print("Nombre de fichiers en erreur :", len(file_errors))
print("\nNombre d'occurrences par balise :")

for tag, count in tag_counter.most_common(30):
    print(tag, ":", count)

Nombre de fichiers analysés : 4036
Nombre de fichiers en erreur : 220

Nombre d'occurrences par balise :
PMID : 53082
CommentsCorrections : 49227
RefSource : 49227
Year : 36399
MeshHeading : 35778
DescriptorName : 35778
Month : 35612
Day : 33398
LastName : 30020
ForeName : 29991
Initials : 29991
Author : 28739
QualifierName : 23216
PubMedPubDate : 20744
AffiliationInfo : 16700
Affiliation : 16700
Keyword : 12366
ArticleId : 11808
Hour : 11798
Minute : 11798
AbstractText : 9492
Chemical : 8192
RegistryNumber : 8192
NameOfSubstance : 8192
PublicationType : 7745
Country : 7021
ELocationID : 4229
Language : 3914
ArticleIdList : 3887
PubDate : 3885


In [6]:
# Nombre de documents contenant chaque balise
important_tags = [
    "PMID",
    "ArticleTitle",
    "AbstractText",
    "Author",
    "Journal",
    "Keyword",
    "PublicationType",
    "Language"
]

for tag in important_tags:

    print(
        f"{tag:20} : {tag_counter[tag]}"
    )

PMID                 : 53082
ArticleTitle         : 3885
AbstractText         : 9492
Author               : 28739
Journal              : 3883
Keyword              : 12366
PublicationType      : 7745
Language             : 3914


In [7]:
from collections import Counter

pmids = []
articles_count = 0
files_without_articles = []
extraction_errors = []

for file in xml_files:
    try:
        tree = etree.parse(str(file))
        root = tree.getroot()

        articles = root.xpath(
            "//*[local-name()='PubmedArticle']"
        )

        if len(articles) == 0:
            files_without_articles.append(file.name)

        for article in articles:
            articles_count += 1

            pmid_elements = article.xpath(
                ".//*[local-name()='PMID']"
            )

            if pmid_elements and pmid_elements[0].text:
                pmids.append(pmid_elements[0].text.strip())

    except Exception as e:
        extraction_errors.append({
            "file": file.name,
            "error": str(e)
        })

unique_pmids = set(pmids)

print("Nombre total de fichiers :", len(xml_files))
print("Nombre total d'articles extraits :", articles_count)
print("Nombre de PMID extraits :", len(pmids))
print("Nombre de PMID uniques :", len(unique_pmids))
print("Nombre de fichiers sans article :", len(files_without_articles))
print("Nombre de fichiers en erreur :", len(extraction_errors))
print("Nombre de PMID dupliqués :", len(pmids) - len(unique_pmids))

Nombre total de fichiers : 4036
Nombre total d'articles extraits : 3883
Nombre de PMID extraits : 3883
Nombre de PMID uniques : 3882
Nombre de fichiers sans article : 0
Nombre de fichiers en erreur : 153
Nombre de PMID dupliqués : 1


In [8]:
from collections import Counter

pmid_counter = Counter(pmids)

duplicates = {
    pmid: count
    for pmid, count in pmid_counter.items()
    if count > 1
}

print("PMID dupliqués :")

for pmid, count in duplicates.items():
    print(f"PMID : {pmid} | Occurrences : {count}")

PMID dupliqués :
PMID : 26560494 | Occurrences : 2


In [9]:
print("Exemples de fichiers en erreur :")

for error in extraction_errors[:10]:
    print("\nFichier :", error["file"])
    print("Erreur :", error["error"][:300])

Exemples de fichiers en erreur :

Fichier : 1070.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1070.xml, line 3)

Fichier : 1071.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1071.xml, line 3)

Fichier : 1072.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1072.xml, line 3)

Fichier : 1073.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1073.xml, line 3)

Fichier : 1074.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1074.xml, line 3)

Fichier : 1075.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1075.xml, line 3)

Fichier : 1076.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1076.xml, line 3)

Fichier : 1077.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1077.xml, line 3)

Fichier : 1078.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1078.xml, line 3)

Fic

In [10]:
from pathlib import Path

files_to_check = ["1070.xml", "1071.xml"]

for filename in files_to_check:
    file_path = DATA_DIR / filename

    print("\n" + "=" * 60)
    print("FICHIER :", filename)

    content = file_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    lines = content.splitlines()

    print("\nDébut du fichier :")
    for i, line in enumerate(lines[:12], start=1):
        print(f"{i}: {line[:200]}")

    print("\nFin du fichier :")
    for i, line in enumerate(lines[-5:], start=max(1, len(lines)-4)):
        print(f"{i}: {line[:200]}")


FICHIER : 1070.xml

Début du fichier :
1: ﻿<PubmedArticleSet>
2: </PubmedArticleSet>
3:     <MedlineCitation Owner="NLM" Status="In-Data-Review">
4:         <PMID Version="1">26578735</PMID>
5:         <DateCreated>
6:             <Year>2016</Year>
7:             <Month>01</Month>
8:             <Day>23</Day>
9:         </DateCreated>
10:         <Article PubModel="Print-Electronic">
11:             <Journal>
12:                 <ISSN IssnType="Electronic">1569-8041</ISSN>

Fin du fichier :
204:     </PubmedData>
205: </PubmedArticle>
206: 
207: 
208: </PubmedArticleSet>

FICHIER : 1071.xml

Début du fichier :
1: ﻿<PubmedArticleSet>
2: </PubmedArticleSet>
3:     <MedlineCitation Owner="NLM" Status="In-Process">
4:         <PMID Version="1">26582381</PMID>
5:         <DateCreated>
6:             <Year>2015</Year>
7:             <Month>12</Month>
8:             <Day>25</Day>
9:         </DateCreated>
10:         <Article PubModel="Print">
11:             <Journal>
12:                 <I

In [11]:
duplicate_pmid = "26560494"

matching_files = []

for file in xml_files:
    try:
        tree = etree.parse(str(file))

        found = tree.xpath(
            f"//*[local-name()='PMID' and text()='{duplicate_pmid}']"
        )

        if found:
            matching_files.append(file.name)

    except Exception:
        continue

print("Fichiers contenant le PMID", duplicate_pmid)

for filename in matching_files:
    print("-", filename)

Fichiers contenant le PMID 26560494
- 1116.xml
- 1117.xml


In [12]:
duplicate_pmid = "26560494"

for filename in ["1116.xml", "1117.xml"]:

    file_path = DATA_DIR / filename

    try:
        tree = etree.parse(str(file_path))

        articles = tree.xpath(
            "//*[local-name()='PubmedArticle']"
        )

        for article in articles:
            pmid = article.xpath(
                ".//*[local-name()='PMID'][1]/text()"
            )

            if pmid and pmid[0].strip() == duplicate_pmid:

                title = article.xpath(
                    ".//*[local-name()='ArticleTitle']/text()"
                )

                abstract = article.xpath(
                    ".//*[local-name()='AbstractText']//text()"
                )

                journal = article.xpath(
                    ".//*[local-name()='Journal']"
                    "//*[local-name()='Title']/text()"
                )

                print("\nFichier :", filename)
                print("PMID :", pmid[0])
                print("Titre :", " ".join(title))
                print("Revue :", journal[0] if journal else "Absent")
                print("Résumé :", " ".join(abstract)[:300])

    except Exception as e:
        print(filename, "Erreur :", e)


Fichier : 1116.xml
PMID : 26560494
Titre : Prospects for the future of prostate cancer vaccines.
Revue : Expert review of vaccines
Résumé : Cancer therapy is undergoing a revolution fueled by clinical data demonstrating that the immune system has significant anti-tumor capability. Although the main focus of this revolution currently rests upon immune checkpoint inhibitors in diseases such as melanoma, lung and bladder cancer, it was act

Fichier : 1117.xml
PMID : 26560494
Titre : Prospects for the future of prostate cancer vaccines.
Revue : Expert review of vaccines
Résumé : Cancer therapy is undergoing a revolution fueled by clinical data demonstrating that the immune system has significant anti-tumor capability. Although the main focus of this revolution currently rests upon immune checkpoint inhibitors in diseases such as melanoma, lung and bladder cancer, it was act


In [13]:
from collections import Counter

error_patterns = Counter()

for error in extraction_errors:
    file_path = DATA_DIR / error["file"]

    with open(file_path, "r", encoding="utf-8-sig", errors="replace") as f:
        content = f.read()

    if "</PubmedArticleSet>" in content:
        position = content.find("</PubmedArticleSet>")
        remaining_content = content[position + len("</PubmedArticleSet>"):].strip()

        if remaining_content:
            error_patterns["Contenu après fermeture de PubmedArticleSet"] += 1
        else:
            error_patterns["Autre problème"] += 1
    else:
        error_patterns["Balise racine absente"] += 1

print("Répartition des anomalies :")

for problem, count in error_patterns.items():
    print(problem, ":", count)

Répartition des anomalies :
Contenu après fermeture de PubmedArticleSet : 55
Balise racine absente : 37
Autre problème : 61


In [15]:
from collections import defaultdict

categories = defaultdict(list)

for error in extraction_errors:
    file_path = DATA_DIR / error["file"]

    with open(file_path, "r", encoding="utf-8-sig", errors="replace") as f:
        content = f.read()

    position = content.find("</PubmedArticleSet>")

    if position != -1 and content[position + len("</PubmedArticleSet>"):].strip():
        categories["Contenu après fermeture"].append(error)
    elif "<PubmedArticleSet" not in content:
        categories["Balise racine absente"].append(error)
    else:
        categories["Autre problème"].append(error)


for category, errors in categories.items():

    print("\n" + "=" * 60)
    print(category, "- Exemples")
    print("=" * 60)

    for error in errors[:3]:
        file_path = DATA_DIR / error["file"]

        with open(file_path, "r", encoding="utf-8-sig", errors="replace") as f:
            lines = f.readlines()

        print("\nFichier :", error["file"])
        print("Erreur :", error["error"])
        print("Début :")

        for line in lines[:5]:
            print(line.rstrip()[:200])

        print("Fin :")

        for line in lines[-5:]:
            print(line.rstrip()[:200])


Contenu après fermeture - Exemples

Fichier : 1070.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1070.xml, line 3)
Début :
<PubmedArticleSet>
</PubmedArticleSet>
    <MedlineCitation Owner="NLM" Status="In-Data-Review">
        <PMID Version="1">26578735</PMID>
        <DateCreated>
Fin :
    </PubmedData>
</PubmedArticle>


</PubmedArticleSet>

Fichier : 1071.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1071.xml, line 3)
Début :
<PubmedArticleSet>
</PubmedArticleSet>
    <MedlineCitation Owner="NLM" Status="In-Process">
        <PMID Version="1">26582381</PMID>
        <DateCreated>
Fin :
    </PubmedData>
</PubmedArticle>


</PubmedArticleSet>

Fichier : 1072.xml
Erreur : Extra content at the end of the document, line 3, column 5 (1072.xml, line 3)
Début :
<PubmedArticleSet>
</PubmedArticleSet>
    <MedlineCitation Owner="NLM" Status="In-Process">
        <PMID Version="1">26582208</PMID>
        <DateCreated>
Fin :
    </Pubmed

In [16]:
file_path = DATA_DIR / "1070.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

print("DÉBUT DU FICHIER")
for i, line in enumerate(lines[:20], start=1):
    print(f"{i}: {line.rstrip()}")

print("\nFIN DU FICHIER")
start = max(0, len(lines) - 15)

for i in range(start, len(lines)):
    print(f"{i+1}: {lines[i].rstrip()}")

DÉBUT DU FICHIER
1: <PubmedArticleSet>
2: </PubmedArticleSet>
3:     <MedlineCitation Owner="NLM" Status="In-Data-Review">
4:         <PMID Version="1">26578735</PMID>
5:         <DateCreated>
6:             <Year>2016</Year>
7:             <Month>01</Month>
8:             <Day>23</Day>
9:         </DateCreated>
10:         <Article PubModel="Print-Electronic">
11:             <Journal>
12:                 <ISSN IssnType="Electronic">1569-8041</ISSN>
13:                 <JournalIssue CitedMedium="Internet">
14:                     <Volume>27</Volume>
15:                     <Issue>2</Issue>
16:                     <PubDate>
17:                         <Year>2016</Year>
18:                         <Month>Feb</Month>
19:                     </PubDate>
20:                 </JournalIssue>

FIN DU FICHIER
194:                 <Hour>6</Hour>
195:                 <Minute>0</Minute>
196:             </PubMedPubDate>
197:         </History>
198:         <PublicationStatus>ppublish</PublicationS

In [ ]:
#reparation d'un seul fichier XML corrompu
from pathlib import Path
from lxml import etree

# 1. Lire le fichier original sans le modifier
file_path = DATA_DIR / "1070.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    content = f.read()

# 2. Supprimer la première fermeture prématurée
premature_close = content.find("</PubmedArticleSet>")

if premature_close == -1:
    raise ValueError("Balise de fermeture introuvable.")

content = (
    content[:premature_close]
    + content[premature_close + len("</PubmedArticleSet>"):]
)

# 3. Ajouter la balise ouvrante PubmedArticle
marker = "<MedlineCitation"
position = content.find(marker)

if position == -1:
    raise ValueError("Balise MedlineCitation introuvable.")

content = content[:position] + "<PubmedArticle>\n    " + content[position:]

# 4. Enregistrer une copie réparée
processed_dir = DATA_DIR.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

repaired_file = processed_dir / "1070_repaired_test.xml"

with open(repaired_file, "w", encoding="utf-8") as f:
    f.write(content)

# 5. Vérifier si le XML est maintenant valide
try:
    tree = etree.parse(str(repaired_file))
    root = tree.getroot()

    print("XML réparé et valide !")

    pmid = root.xpath(
        "//*[local-name()='MedlineCitation']/*[local-name()='PMID']/text()"
    )

    print("PMID :", pmid[0] if pmid else "Non trouvé")

except etree.XMLSyntaxError as e:
    print("Erreur XML :", e)

XML réparé et valide !
PMID : 26578735


In [18]:
from pathlib import Path
from lxml import etree

repair_candidates = []
other_structures = []

for error in categories["Contenu après fermeture"]:

    file_path = DATA_DIR / error["file"]

    with open(file_path, "r", encoding="utf-8-sig") as f:
        content = f.read()

    # Vérifier la fermeture prématurée
    first_close = content.find("</PubmedArticleSet>")

    # Vérifier la présence de MedlineCitation après cette fermeture
    if first_close != -1:
        remaining = content[first_close + len("</PubmedArticleSet>"):]

        has_medline = "<MedlineCitation" in remaining
        has_article_open = "<PubmedArticle>" in content[:first_close]

        if has_medline and not has_article_open:
            repair_candidates.append(error["file"])
        else:
            other_structures.append(error["file"])

print("Fichiers candidats à la même réparation :", len(repair_candidates))
print("Fichiers avec une structure différente :", len(other_structures))

print("\nExemples de fichiers candidats :")
for filename in repair_candidates[:10]:
    print("-", filename)

print("\nExemples de structures différentes :")
for filename in other_structures[:10]:
    print("-", filename)

Fichiers candidats à la même réparation : 20
Fichiers avec une structure différente : 35

Exemples de fichiers candidats :
- 1070.xml
- 1071.xml
- 1072.xml
- 1073.xml
- 1074.xml
- 1075.xml
- 1076.xml
- 1077.xml
- 1078.xml
- 1079.xml

Exemples de structures différentes :
- 1095.xml
- 1099.xml
- 1126.xml
- 1138.xml
- 1142.xml
- 1148.xml
- 1151.xml
- 1154.xml
- 1159.xml
- 1166.xml


In [19]:
for filename in other_structures[:3]:

    file_path = DATA_DIR / filename

    with open(file_path, "r", encoding="utf-8-sig") as f:
        content = f.read()

    lines = content.splitlines()

    print("\n" + "=" * 60)
    print("FICHIER :", filename)
    print("=" * 60)

    print("\nNombre de balises importantes :")
    for tag in [
        "<PubmedArticleSet",
        "</PubmedArticleSet>",
        "<PubmedArticle",
        "</PubmedArticle>",
        "<MedlineCitation"
    ]:
        print(tag, ":", content.count(tag))

    print("\nDébut du fichier :")
    for line in lines[:8]:
        print(line[:180])

    print("\nFin du fichier :")
    for line in lines[-8:]:
        print(line[:180])


FICHIER : 1095.xml

Nombre de balises importantes :
<PubmedArticleSet : 1
</PubmedArticleSet> : 1
<PubmedArticle : 3
</PubmedArticle> : 2
<MedlineCitation : 2

Début du fichier :
<PubmedArticleSet>
<PubmedArticle>
    <MedlineCitation Owner="NLM" Status="In-Process">
        <PMID Version="1">26569573</PMID>
        <DateCreated>
            <Year>2015</Year>
            <Month>12</Month>
            <Day>28</Day>

Fin du fichier :
            <ArticleId IdType="doi">10.3390/molecules201119685</ArticleId>
            <ArticleId IdType="pubmed">26569200</ArticleId>
        </ArticleIdList>
    </PubmedData>
</PubmedArticle>


<PubmedArticle>

FICHIER : 1099.xml

Nombre de balises importantes :
<PubmedArticleSet : 1
</PubmedArticleSet> : 1
<PubmedArticle : 3
</PubmedArticle> : 2
<MedlineCitation : 2

Début du fichier :
<PubmedArticleSet>
<PubmedArticle>
    <MedlineCitation Owner="NLM" Status="In-Data-Review">
        <PMID Version="1">26568444</PMID>
        <DateCreated>
            <

In [20]:
import re

file_path = DATA_DIR / "1095.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    content = f.read()

tags = re.findall(
    r"</?(?:PubmedArticleSet|PubmedArticle|MedlineCitation|PubmedData)\b[^>]*>",
    content
)

for i, tag in enumerate(tags, start=1):
    print(i, tag)

1 <PubmedArticleSet>
2 <PubmedArticle>
3 <MedlineCitation Owner="NLM" Status="In-Process">
4 </MedlineCitation>
5 <PubmedData>
6 </PubmedData>
7 </PubmedArticle>
8 </PubmedArticleSet>
9 <MedlineCitation Owner="NLM" Status="In-Process">
10 </MedlineCitation>
11 <PubmedData>
12 </PubmedData>
13 </PubmedArticle>
14 <PubmedArticle>


In [21]:
import re

file_path = DATA_DIR / "1095.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

pattern = re.compile(
    r"</?(?:PubmedArticleSet|PubmedArticle|MedlineCitation|PubmedData)\b"
)

for i, line in enumerate(lines, start=1):
    if pattern.search(line):
        print(f"Ligne {i}: {line.strip()}")

Ligne 1: <PubmedArticleSet>
Ligne 2: <PubmedArticle>
Ligne 3: <MedlineCitation Owner="NLM" Status="In-Process">
Ligne 99: </MedlineCitation>
Ligne 100: <PubmedData>
Ligne 150: </PubmedData>
Ligne 151: </PubmedArticle>
Ligne 154: </PubmedArticleSet>
Ligne 155: <MedlineCitation Owner="NLM" Status="In-Process">
Ligne 272: </MedlineCitation>
Ligne 273: <PubmedData>
Ligne 318: </PubmedData>
Ligne 319: </PubmedArticle>
Ligne 322: <PubmedArticle>


In [22]:
file_path = DATA_DIR / "1095.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

print("À PARTIR DE LA LIGNE 315 :")

for i in range(314, len(lines)):
    print(f"Ligne {i+1}: {lines[i].rstrip()}")

À PARTIR DE LA LIGNE 315 :
Ligne 315:             <ArticleId IdType="doi">10.3390/molecules201119685</ArticleId>
Ligne 316:             <ArticleId IdType="pubmed">26569200</ArticleId>
Ligne 317:         </ArticleIdList>
Ligne 318:     </PubmedData>
Ligne 319: </PubmedArticle>
Ligne 320: 
Ligne 321: 
Ligne 322: <PubmedArticle>


In [23]:
file_path = DATA_DIR / "1095.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    content = f.read()

print("Nombre de caractères :", len(content))
print("Nombre de balises MedlineCitation :", content.count("<MedlineCitation"))
print("Nombre de balises PubmedData :", content.count("<PubmedData>"))
print("Nombre de balises PMID :", content.count("<PMID"))
print("Contenu après le dernier PubmedArticle :")
print(repr(content[content.rfind("<PubmedArticle>"):]))

Nombre de caractères : 19039
Nombre de balises MedlineCitation : 2
Nombre de balises PubmedData : 2
Nombre de balises PMID : 2
Contenu après le dernier PubmedArticle :
'<PubmedArticle>\n'


In [ ]:
from pathlib import Path
from lxml import etree

# 1. Lire le fichier original
file_path = DATA_DIR / "1095.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    content = f.read()

# 2. Supprimer la première fermeture prématurée
first_close = content.find("</PubmedArticleSet>")

if first_close == -1:
    raise ValueError("Balise </PubmedArticleSet> introuvable.")

content = (
    content[:first_close]
    + content[first_close + len("</PubmedArticleSet>"):]
)

# 3. Ajouter PubmedArticle avant le deuxième MedlineCitation
marker = "<MedlineCitation"

first_position = content.find(marker)

if first_position == -1:
    raise ValueError("Première balise MedlineCitation introuvable.")

second_position = content.find(
    marker,
    first_position + len(marker)
)

if second_position == -1:
    raise ValueError("Deuxième balise MedlineCitation introuvable.")

content = (
    content[:second_position]
    + "<PubmedArticle>\n"
    + content[second_position:]
)

# 4. Supprimer la dernière balise PubmedArticle vide
content = content.rstrip()

if content.endswith("<PubmedArticle>"):
    content = content[:-len("<PubmedArticle>")].rstrip()

# 5. Ajouter la fermeture finale de la racine
if not content.endswith("</PubmedArticleSet>"):
    content += "\n</PubmedArticleSet>"

# 6. Créer le dossier processed
processed_dir = DATA_DIR.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 7. Enregistrer le fichier réparé
repaired_file = processed_dir / "1095_repaired_test.xml"

with open(repaired_file, "w", encoding="utf-8") as f:
    f.write(content)

# 8. Vérifier la validité du XML
try:
    tree = etree.parse(str(repaired_file))
    root = tree.getroot()

    articles = root.xpath(
        "./*[local-name()='PubmedArticle']"
    )

    print("XML réparé et valide !")
    print("Nombre d'articles :", len(articles))

    print("\nPMID des articles :")

    for article in articles:
        pmid = article.xpath(
            "./*[local-name()='MedlineCitation']/*[local-name()='PMID']/text()"
        )

        print("-", pmid[0] if pmid else "PMID absent")

    print("\nFichier enregistré :", repaired_file)

except etree.XMLSyntaxError as e:
    print("Erreur XML :", e)

XML réparé et valide !
Nombre d'articles : 2

PMID des articles :
- 26569573
- 26569200

Fichier enregistré : C:\Users\Ghada\Desktop\medical-data-warehouse\data\processed\1095_repaired_test.xml


In [26]:
from pathlib import Path
from lxml import etree

# 1. Lire le fichier original
file_path = DATA_DIR / "1099.xml"

with open(file_path, "r", encoding="utf-8-sig") as f:
    content = f.read()

# 2. Supprimer la première fermeture prématurée
first_close = content.find("</PubmedArticleSet>")

if first_close == -1:
    raise ValueError("Balise </PubmedArticleSet> introuvable.")

content = (
    content[:first_close]
    + content[first_close + len("</PubmedArticleSet>"):]
)

# 3. Ajouter PubmedArticle avant le deuxième MedlineCitation
marker = "<MedlineCitation"

first_position = content.find(marker)

if first_position == -1:
    raise ValueError("Première balise MedlineCitation introuvable.")

second_position = content.find(
    marker,
    first_position + len(marker)
)

if second_position == -1:
    raise ValueError("Deuxième balise MedlineCitation introuvable.")

content = (
    content[:second_position]
    + "<PubmedArticle>\n"
    + content[second_position:]
)

# 4. Supprimer la dernière balise PubmedArticle vide
content = content.rstrip()

if content.endswith("<PubmedArticle>"):
    content = content[:-len("<PubmedArticle>")].rstrip()

# 5. Ajouter la fermeture finale de la racine
if not content.endswith("</PubmedArticleSet>"):
    content += "\n</PubmedArticleSet>"

# 6. Créer le dossier processed
processed_dir = DATA_DIR.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 7. Enregistrer le fichier réparé
repaired_file = processed_dir / "1095_repaired_test.xml"

with open(repaired_file, "w", encoding="utf-8") as f:
    f.write(content)

# 8. Vérifier la validité du XML
try:
    tree = etree.parse(str(repaired_file))
    root = tree.getroot()

    articles = root.xpath(
        "./*[local-name()='PubmedArticle']"
    )

    print("XML réparé et valide !")
    print("Nombre d'articles :", len(articles))

    print("\nPMID des articles :")

    for article in articles:
        pmid = article.xpath(
            "./*[local-name()='MedlineCitation']/*[local-name()='PMID']/text()"
        )

        print("-", pmid[0] if pmid else "PMID absent")

    print("\nFichier enregistré :", repaired_file)

except etree.XMLSyntaxError as e:
    print("Erreur XML :", e)

XML réparé et valide !
Nombre d'articles : 2

PMID des articles :
- 26568444
- 26568368

Fichier enregistré : C:\Users\Ghada\Desktop\medical-data-warehouse\data\processed\1095_repaired_test.xml


In [27]:

import re
import shutil
import pandas as pd
from pathlib import Path
from lxml import etree

# ==========================================
# 1. CONFIGURATION
# ==========================================

processed_dir = DATA_DIR.parent / "processed"
rejected_dir = DATA_DIR.parent / "rejected"
reports_dir = DATA_DIR.parent.parent / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
rejected_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Les 55 fichiers de la catégorie étudiée
files_to_process = categories["Contenu après fermeture"]

results = []


# ==========================================
# 2. FONCTION DE RÉPARATION
# ==========================================

def repair_xml(file_path):

    with open(file_path, "r", encoding="utf-8-sig") as f:
        content = f.read()

    original_content = content

    # Supprimer les fermetures prématurées de la racine
    root_closes = list(re.finditer(
        r"</PubmedArticleSet\s*>",
        content
    ))

    if len(root_closes) > 1:
        for match in reversed(root_closes[:-1]):
            content = (
                content[:match.start()]
                + content[match.end():]
            )

    # Ajouter la fermeture finale si elle manque
    if not re.search(r"</PubmedArticleSet\s*>\s*$", content):
        content = content.rstrip() + "\n</PubmedArticleSet>\n"

    # Supprimer un dernier article vide :
    # une balise ouvrante sans MedlineCitation ni contenu
    root_close_pos = content.rfind("</PubmedArticleSet>")
    last_open = content.rfind("<PubmedArticle>")
    last_close = content.rfind("</PubmedArticle>")

    if last_open > last_close and last_open < root_close_pos:
        between = content[last_open + len("<PubmedArticle>"):root_close_pos]

        if not between.strip():
            content = (
                content[:last_open]
                + content[last_open + len("<PubmedArticle>"):]
            )

    # Repérer les balises structurelles
    pattern = re.compile(
        r"<(/?)PubmedArticle(?=[\s>])[^>]*>|"
        r"<MedlineCitation\b[^>]*>|"
        r"</PubmedArticleSet\s*>"
    )

    output = []
    position = 0
    article_open = False

    for match in pattern.finditer(content):

        token = match.group(0)

        output.append(content[position:match.start()])

        # Fermeture de la racine
        if token.startswith("</PubmedArticleSet"):

            if article_open:
                output.append("</PubmedArticle>")
                article_open = False

            output.append(token)

        # Balise PubmedArticle ouvrante ou fermante
        elif token.startswith("<PubmedArticle"):

            if token.startswith("</"):

                if article_open:
                    output.append(token)
                    article_open = False
                else:
                    raise ValueError(
                        "Balise PubmedArticle fermante sans ouverture"
                    )

            else:

                if article_open:
                    output.append("</PubmedArticle>")

                output.append(token)
                article_open = True

        # MedlineCitation sans article parent
        elif token.startswith("<MedlineCitation"):

            if not article_open:
                output.append("<PubmedArticle>")
                article_open = True

            output.append(token)

        position = match.end()

    output.append(content[position:])

    if article_open:
        output.append("</PubmedArticle>")

    repaired_content = "".join(output)

    # Vérification XML stricte
    parser = etree.XMLParser(recover=False)

    root = etree.fromstring(
        repaired_content.encode("utf-8"),
        parser
    )

    articles = root.xpath(
        "./*[local-name()='PubmedArticle']"
    )

    # Vérifier que chaque article possède un PMID principal
    pmids = []

    for article in articles:

        pmid = article.xpath(
            "./*[local-name()='MedlineCitation']/"
            "*[local-name()='PMID']/text()"
        )

        if not pmid:
            raise ValueError(
                "Un article ne possède pas de PMID principal"
            )

        pmids.append(pmid[0].strip())

    if len(pmids) != len(set(pmids)):
        raise ValueError("PMID dupliqués dans le fichier")

    return repaired_content, len(articles), pmids


# ==========================================
# 3. TRAITEMENT AUTOMATIQUE DES FICHIERS
# ==========================================

for error in files_to_process:

    filename = error["file"]
    file_path = DATA_DIR / filename

    try:

        repaired_content, article_count, pmids = repair_xml(file_path)

        output_file = processed_dir / f"{file_path.stem}_repaired.xml"

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(repaired_content)

        results.append({
            "file": filename,
            "status": "Réparé",
            "articles": article_count,
            "pmids": ", ".join(pmids),
            "error": ""
        })

    except Exception as e:

        results.append({
            "file": filename,
            "status": "À vérifier",
            "articles": 0,
            "pmids": "",
            "error": str(e)
        })


# ==========================================
# 4. GÉNÉRER LE RAPPORT
# ==========================================

report = pd.DataFrame(results)

report_path = reports_dir / "xml_repair_report.csv"

report.to_csv(
    report_path,
    index=False,
    encoding="utf-8-sig"
)

print("TRAITEMENT TERMINÉ")
print("==============================")
print("Fichiers examinés :", len(report))
print("Fichiers réparés :", (report["status"] == "Réparé").sum())
print("Fichiers à vérifier :", (report["status"] == "À vérifier").sum())
print("Articles récupérés :", report["articles"].sum())

print("\nRapport enregistré :", report_path)

display(report)


TRAITEMENT TERMINÉ
Fichiers examinés : 55
Fichiers réparés : 21
Fichiers à vérifier : 34
Articles récupérés : 21

Rapport enregistré : C:\Users\Ghada\Desktop\medical-data-warehouse\reports\xml_repair_report.csv


,file,status,articles,pmids,error
0,1070.xml,Réparé,1,26578735,
1,1071.xml,Réparé,1,26582381,
2,1072.xml,Réparé,1,26582208,
3,1073.xml,Réparé,1,26582203,
4,1074.xml,Réparé,1,26581688,
5,1075.xml,Réparé,1,26581400,
6,1076.xml,Réparé,1,26581226,
7,1077.xml,Réparé,1,26581128,
8,1078.xml,Réparé,1,26578786,
9,1079.xml,Réparé,1,26573846,


In [29]:

from pathlib import Path
from lxml import etree
import pandas as pd
import shutil

# ==========================================
# 1. Définir les chemins du projet
# ==========================================


from pathlib import Path

# Remonter du dossier notebooks vers la racine du projet
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REJECTED_DIR = PROJECT_DIR / "data" / "rejected"
REPORT_DIR = PROJECT_DIR / "reports"

# Créer les dossiers nécessaires
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Vérifier les chemins
print("Racine du projet :", PROJECT_DIR)
print("Dossier raw :", RAW_DIR)
print("Dossier raw existe :", RAW_DIR.exists())


# ==========================================
# 2. Fonction de validation XML
# ==========================================

def validate_xml(file_path):
    """
    Vérifie que le XML est valide et que chaque
    PubmedArticle possède un PMID principal.
    """

    try:
        parser = etree.XMLParser(recover=False)
        tree = etree.parse(str(file_path), parser)

        articles = tree.xpath(
            "//*[local-name()='PubmedArticle']"
        )

        if len(articles) == 0:
            return False, 0, "Aucun article PubmedArticle"

        for article in articles:
            pmids = article.xpath(
                "./*[local-name()='MedlineCitation']"
                "/*[local-name()='PMID']/text()"
            )

            if not pmids or not pmids[0].strip():
                return False, len(articles), "PMID principal manquant"

        return True, len(articles), "XML valide"

    except Exception as e:
        return False, 0, str(e)


# ==========================================
# 3. Traiter tous les fichiers originaux
# ==========================================

xml_files = sorted(RAW_DIR.glob("*.xml"))

report = []

for i, original_file in enumerate(xml_files, start=1):

    filename = original_file.name
    print(f"[{i}/{len(xml_files)}] Traitement : {filename}")

    # Vérifier d'abord le fichier original
    is_valid, nb_articles, message = validate_xml(original_file)

    if is_valid:

        # Original valide : le copier dans processed
        destination = PROCESSED_DIR / filename
        shutil.copy2(original_file, destination)

        status = "Original valide"
        source_used = filename

    else:

        # Chercher une version réparée
        repaired_candidates = [
            PROCESSED_DIR / f"{original_file.stem}_repaired.xml",
            PROCESSED_DIR / f"{original_file.stem}_repaired_test.xml"
        ]

        repaired_file = None

        for candidate in repaired_candidates:
            if candidate.exists():
                repaired_file = candidate
                break

        if repaired_file is not None:

            repaired_valid, repaired_count, repaired_message = (
                validate_xml(repaired_file)
            )

            if repaired_valid:

                # Copier la version réparée avec le nom original
                destination = PROCESSED_DIR / filename
                shutil.copy2(repaired_file, destination)

                status = "Réparé valide"
                source_used = repaired_file.name
                nb_articles = repaired_count
                message = repaired_message

            else:

                # Version réparée invalide : rejet
                destination = REJECTED_DIR / filename
                shutil.copy2(original_file, destination)

                status = "Rejeté"
                source_used = filename
                message = "Version réparée invalide : " + repaired_message

        else:

            # Aucune réparation disponible : rejet
            destination = REJECTED_DIR / filename
            shutil.copy2(original_file, destination)

            status = "Rejeté"
            source_used = filename

    report.append({
        "Fichier": filename,
        "Statut": status,
        "Nombre_articles": nb_articles,
        "Source_utilisee": source_used,
        "Remarque": message
    })


# ==========================================
# 4. Générer le rapport CSV
# ==========================================

df_report = pd.DataFrame(report)

report_path = REPORT_DIR / "final_xml_classification.csv"

df_report.to_csv(
    report_path,
    index=False,
    encoding="utf-8-sig"
)


# ==========================================
# 5. Afficher le bilan
# ==========================================

print("\n========== BILAN FINAL ==========")

print("Nombre de fichiers originaux :", len(xml_files))

print("\nRépartition par statut :")
print(df_report["Statut"].value_counts())

print("\nNombre total d'articles valides :",
      df_report.loc[
          df_report["Statut"].isin(
              ["Original valide", "Réparé valide"]
          ),
          "Nombre_articles"
      ].sum())

print("\nDossier des fichiers valides :", PROCESSED_DIR)
print("Dossier des fichiers rejetés :", REJECTED_DIR)
print("Rapport CSV :", report_path)

print("\nClassement terminé !")


Racine du projet : c:\Users\Ghada\Desktop\medical-data-warehouse
Dossier raw : c:\Users\Ghada\Desktop\medical-data-warehouse\data\raw
Dossier raw existe : True
[1/4036] Traitement : 1000.xml
[2/4036] Traitement : 1001.xml
[3/4036] Traitement : 1002.xml
[4/4036] Traitement : 1003.xml
[5/4036] Traitement : 1004.xml
[6/4036] Traitement : 1005.xml
[7/4036] Traitement : 1006.xml
[8/4036] Traitement : 1007.xml
[9/4036] Traitement : 1008.xml
[10/4036] Traitement : 1009.xml
[11/4036] Traitement : 1010.xml
[12/4036] Traitement : 1011.xml
[13/4036] Traitement : 1012.xml
[14/4036] Traitement : 1013.xml
[15/4036] Traitement : 1014.xml
[16/4036] Traitement : 1015.xml
[17/4036] Traitement : 1016.xml
[18/4036] Traitement : 1017.xml
[19/4036] Traitement : 1018.xml
[20/4036] Traitement : 1019.xml
[21/4036] Traitement : 1020.xml
[22/4036] Traitement : 1021.xml
[23/4036] Traitement : 1022.xml
[24/4036] Traitement : 1023.xml
[25/4036] Traitement : 1024.xml
[26/4036] Traitement : 1025.xml
[27/4036] Traitem

In [30]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
REPORT_PATH = PROJECT_DIR / "reports" / "final_xml_classification.csv"

df_report = pd.read_csv(REPORT_PATH)

print("========== BILAN DE CLASSIFICATION ==========")

print("\nNombre total de fichiers :", len(df_report))

print("\nRépartition des fichiers :")
print(df_report["Statut"].value_counts().to_string())

print("\nNombre total d'articles valides :")
print(
    df_report.loc[
        df_report["Statut"].isin(
            ["Original valide", "Réparé valide"]
        ),
        "Nombre_articles"
    ].sum()
)

print("\nNombre de fichiers rejetés :",
      (df_report["Statut"] == "Rejeté").sum())

print("\nNombre de fichiers dans processed :",
      len(list((PROJECT_DIR / "data" / "processed").glob("*.xml"))))

print("\nNombre de fichiers dans rejected :",
      len(list((PROJECT_DIR / "data" / "rejected").glob("*.xml"))))


========== BILAN DE CLASSIFICATION ==========

Nombre total de fichiers : 4036

Répartition des fichiers :
Statut
Original valide    3883
Rejeté              131
Réparé valide        22

Nombre total d'articles valides :
3906

Nombre de fichiers rejetés : 131

Nombre de fichiers dans processed : 3928

Nombre de fichiers dans rejected : 131


In [31]:

rejected = df_report[df_report["Statut"] == "Rejeté"]

print("Nombre de fichiers rejetés :", len(rejected))

display(
    rejected[
        ["Fichier", "Remarque"]
    ].head(20)
)


Nombre de fichiers rejetés : 131


,Fichier,Remarque
99,1099.xml,"Extra content at the end of the document, line..."
108,1108.xml,Premature end of data in tag PubMedPubDate lin...
120,1120.xml,Premature end of data in tag PubmedArticle lin...
126,1126.xml,"Extra content at the end of the document, line..."
138,1138.xml,"Extra content at the end of the document, line..."
142,1142.xml,"Extra content at the end of the document, line..."
144,1144.xml,Premature end of data in tag ArticleIdList lin...
148,1148.xml,"Extra content at the end of the document, line..."
151,1151.xml,"Extra content at the end of the document, line..."
154,1154.xml,"Extra content at the end of the document, line..."


In [32]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_PATH = PROJECT_DIR / "reports" / "final_xml_classification.csv"

df = pd.read_csv(REPORT_PATH)

# Fichiers qui devraient être dans processed
expected_files = set(
    df[df["Statut"].isin(
        ["Original valide", "Réparé valide"]
    )]["Fichier"].tolist()
)

# Fichiers actuellement dans processed
actual_files = {
    f.name for f in PROCESSED_DIR.glob("*.xml")
}

# Comparaison
extra_files = sorted(actual_files - expected_files)
missing_files = sorted(expected_files - actual_files)

print("Fichiers attendus :", len(expected_files))
print("Fichiers présents :", len(actual_files))
print("Fichiers supplémentaires :", len(extra_files))
print("Fichiers manquants :", len(missing_files))

print("\n========== FICHIERS SUPPLÉMENTAIRES ==========")
for f in extra_files:
    print(f)

print("\n========== FICHIERS MANQUANTS ==========")
for f in missing_files:
    print(f)


Fichiers attendus : 3905
Fichiers présents : 3928
Fichiers supplémentaires : 23
Fichiers manquants : 0

========== FICHIERS SUPPLÉMENTAIRES ==========
1070_repaired.xml
1070_repaired_test.xml
1071_repaired.xml
1072_repaired.xml
1073_repaired.xml
1074_repaired.xml
1075_repaired.xml
1076_repaired.xml
1077_repaired.xml
1078_repaired.xml
1079_repaired.xml
1080_repaired.xml
1081_repaired.xml
1082_repaired.xml
1083_repaired.xml
1084_repaired.xml
1085_repaired.xml
1086_repaired.xml
1087_repaired.xml
1088_repaired.xml
1089_repaired.xml
1095_repaired_test.xml
1924_repaired.xml

========== FICHIERS MANQUANTS ==========


In [33]:

from pathlib import Path
import shutil

PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ARCHIVE_DIR = PROJECT_DIR / "data" / "repair_archive"

ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

# Identifier les fichiers intermédiaires
extra_files = [
    f for f in PROCESSED_DIR.glob("*.xml")
    if "_repaired" in f.stem
]

print("Fichiers intermédiaires trouvés :", len(extra_files))

# Déplacer les fichiers intermédiaires vers l'archive
for file in extra_files:
    destination = ARCHIVE_DIR / file.name

    if destination.exists():
        print("Déjà archivé :", file.name)
        continue

    shutil.move(str(file), str(destination))
    print("Archivé :", file.name)

print("\nClassement des fichiers intermédiaires terminé !")


Fichiers intermédiaires trouvés : 23
Archivé : 1070_repaired.xml
Archivé : 1070_repaired_test.xml
Archivé : 1071_repaired.xml
Archivé : 1072_repaired.xml
Archivé : 1073_repaired.xml
Archivé : 1074_repaired.xml
Archivé : 1075_repaired.xml
Archivé : 1076_repaired.xml
Archivé : 1077_repaired.xml
Archivé : 1078_repaired.xml
Archivé : 1079_repaired.xml
Archivé : 1080_repaired.xml
Archivé : 1081_repaired.xml
Archivé : 1082_repaired.xml
Archivé : 1083_repaired.xml
Archivé : 1084_repaired.xml
Archivé : 1085_repaired.xml
Archivé : 1086_repaired.xml
Archivé : 1087_repaired.xml
Archivé : 1088_repaired.xml
Archivé : 1089_repaired.xml
Archivé : 1095_repaired_test.xml
Archivé : 1924_repaired.xml

Classement des fichiers intermédiaires terminé !


In [34]:

print("Fichiers XML finaux dans processed :",
      len(list(PROCESSED_DIR.glob("*.xml"))))

print("Fichiers conservés dans repair_archive :",
      len(list(ARCHIVE_DIR.glob("*.xml"))))


Fichiers XML finaux dans processed : 3905
Fichiers conservés dans repair_archive : 23


In [35]:

from pathlib import Path
from lxml import etree
import pandas as pd

# ==========================================
# 1. Définir les chemins
# ==========================================

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. Extraire les PMID principaux
# ==========================================

articles_data = []
missing_pmids = []
errors = []

xml_files = sorted(PROCESSED_DIR.glob("*.xml"))

for file in xml_files:

    try:
        tree = etree.parse(str(file))

        articles = tree.xpath(
            "//*[local-name()='PubmedArticle']"
        )

        for index, article in enumerate(articles, start=1):

            # Extraire uniquement le PMID principal
            # situé directement dans MedlineCitation
            pmid_values = article.xpath(
                "./*[local-name()='MedlineCitation']"
                "/*[local-name()='PMID']/text()"
            )

            pmid = (
                pmid_values[0].strip()
                if pmid_values and pmid_values[0].strip()
                else None
            )

            if pmid is None:
                missing_pmids.append({
                    "Fichier": file.name,
                    "Numero_article": index
                })
            else:
                articles_data.append({
                    "Fichier": file.name,
                    "Numero_article": index,
                    "PMID": pmid
                })

    except Exception as e:
        errors.append({
            "Fichier": file.name,
            "Erreur": str(e)
        })

# ==========================================
# 3. Créer le DataFrame
# ==========================================

df_articles = pd.DataFrame(articles_data)

df_missing = pd.DataFrame(
    missing_pmids,
    columns=["Fichier", "Numero_article"]
)

df_errors = pd.DataFrame(
    errors,
    columns=["Fichier", "Erreur"]
)

# ==========================================
# 4. Détecter les PMID dupliqués
# ==========================================

if not df_articles.empty:

    duplicates = df_articles[
        df_articles.duplicated(
            subset=["PMID"],
            keep=False
        )
    ].sort_values("PMID")

else:
    duplicates = pd.DataFrame(
        columns=["Fichier", "Numero_article", "PMID"]
    )

# ==========================================
# 5. Sauvegarder les résultats
# ==========================================

df_articles.to_csv(
    REPORT_DIR / "articles_pmids.csv",
    index=False,
    encoding="utf-8-sig"
)

duplicates.to_csv(
    REPORT_DIR / "duplicate_pmids.csv",
    index=False,
    encoding="utf-8-sig"
)

df_missing.to_csv(
    REPORT_DIR / "missing_pmids.csv",
    index=False,
    encoding="utf-8-sig"
)

df_errors.to_csv(
    REPORT_DIR / "pmid_extraction_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================
# 6. Afficher le bilan
# ==========================================

print("========== BILAN EXTRACTION PMID ==========")

print("Fichiers XML traités :", len(xml_files))
print("Articles identifiés :", len(df_articles) + len(df_missing))
print("Articles avec PMID :", len(df_articles))
print("Articles sans PMID :", len(df_missing))

print("Nombre de PMID uniques :",
      df_articles["PMID"].nunique() if not df_articles.empty else 0)

print("Nombre de PMID dupliqués :",
      duplicates["PMID"].nunique() if not duplicates.empty else 0)

print("Nombre de lignes concernées par les doublons :",
      len(duplicates))

print("Erreurs de lecture :", len(df_errors))

print("\n========== APERÇU DES DOUBLONS ==========")
display(duplicates.head(20))

print("\n========== ARTICLES SANS PMID ==========")
display(df_missing.head(20))

print("\nRapports enregistrés dans :", REPORT_DIR)


========== BILAN EXTRACTION PMID ==========
Fichiers XML traités : 3905
Articles identifiés : 3906
Articles avec PMID : 3906
Articles sans PMID : 0
Nombre de PMID uniques : 3903
Nombre de PMID dupliqués : 3
Nombre de lignes concernées par les doublons : 6
Erreurs de lecture : 0

========== APERÇU DES DOUBLONS ==========


,Fichier,Numero_article,PMID
852,1924.xml,1,26165617
869,1941.xml,1,26165617
115,1116.xml,1,26560494
116,1117.xml,1,26560494
95,1095.xml,1,26568444
99,1098.xml,1,26568444



========== ARTICLES SANS PMID ==========


,Fichier,Numero_article



Rapports enregistrés dans : c:\Users\Ghada\Desktop\medical-data-warehouse\reports


In [36]:

from pathlib import Path
from lxml import etree
import pandas as pd
import hashlib

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

# Charger les articles contenant les PMID dupliqués
df_articles = pd.read_csv(REPORT_DIR / "articles_pmids.csv")

duplicate_pmids = (
    df_articles["PMID"].value_counts()
)

duplicate_pmids = duplicate_pmids[
    duplicate_pmids > 1
].index.tolist()

comparison_results = []

for pmid in duplicate_pmids:

    rows = df_articles[df_articles["PMID"] == pmid]

    for _, row in rows.iterrows():

        file_path = PROCESSED_DIR / row["Fichier"]

        tree = etree.parse(str(file_path))

        articles = tree.xpath(
            "//*[local-name()='PubmedArticle']"
        )

        for article in articles:

            pmid_values = article.xpath(
                "./*[local-name()='MedlineCitation']"
                "/*[local-name()='PMID']/text()"
            )

            if not pmid_values or pmid_values[0].strip() != str(pmid):
                continue

            # Comparer le contenu XML normalisé
            xml_content = etree.tostring(
                article,
                method="c14n"
            )

            content_hash = hashlib.sha256(
                xml_content
            ).hexdigest()

            # Extraire quelques informations utiles
            title_values = article.xpath(
                ".//*[local-name()='ArticleTitle']"
            )

            title = (
                "".join(title_values[0].itertext()).strip()
                if title_values else "Titre absent"
            )

            journal_values = article.xpath(
                ".//*[local-name()='Journal']"
                "/*[local-name()='Title']/text()"
            )

            journal = (
                journal_values[0]
                if journal_values else "Journal absent"
            )

            comparison_results.append({
                "PMID": pmid,
                "Fichier": row["Fichier"],
                "Titre": title,
                "Journal": journal,
                "Hash_XML": content_hash
            })

            break

# Créer le tableau de comparaison
df_comparison = pd.DataFrame(comparison_results)

# Déterminer si les contenus sont identiques
summary = (
    df_comparison.groupby("PMID")
    .agg(
        Nombre_fichiers=("Fichier", "count"),
        Nombre_contenus_differents=("Hash_XML", "nunique")
    )
    .reset_index()
)

summary["Resultat"] = summary[
    "Nombre_contenus_differents"
].apply(
    lambda x: "Contenus identiques"
    if x == 1
    else "Contenus différents"
)

# Sauvegarder les résultats
df_comparison.to_csv(
    REPORT_DIR / "duplicate_articles_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    REPORT_DIR / "duplicate_articles_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("========== COMPARAISON DES DOUBLONS ==========")
display(summary)

print("\n========== DÉTAILS DES ARTICLES ==========")
display(
    df_comparison[
        ["PMID", "Fichier", "Titre", "Journal"]
    ]
)


========== COMPARAISON DES DOUBLONS ==========


,PMID,Nombre_fichiers,Nombre_contenus_differents,Resultat
0,26165617,2,2,Contenus différents
1,26560494,2,1,Contenus identiques
2,26568444,2,1,Contenus identiques



========== DÉTAILS DES ARTICLES ==========


,PMID,Fichier,Titre,Journal
0,26560494,1116.xml,Prospects for the future of prostate cancer va...,Expert review of vaccines
1,26560494,1117.xml,Prospects for the future of prostate cancer va...,Expert review of vaccines
2,26568444,1095.xml,Which patients benefit from post-implant CT do...,Journal of medical imaging and radiation oncology
3,26568444,1098.xml,Which patients benefit from post-implant CT do...,Journal of medical imaging and radiation oncology
4,26165617,1924.xml,Orbital Compartment Syndrome as the Presenting...,Urology
5,26165617,1941.xml,Orbital Compartment Syndrome as the Presenting...,Urology


In [37]:

from pathlib import Path
from lxml import etree
import pandas as pd
import hashlib

# ==========================================
# 1. Définir les chemins
# ==========================================

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CLEANED_DIR = PROJECT_DIR / "data" / "cleaned"
REPORT_DIR = PROJECT_DIR / "reports"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. Extraire les articles
# ==========================================

records = []

for file in sorted(PROCESSED_DIR.glob("*.xml")):

    tree = etree.parse(str(file))

    articles = tree.xpath(
        "//*[local-name()='PubmedArticle']"
    )

    for index, article in enumerate(articles, start=1):

        pmid_values = article.xpath(
            "./*[local-name()='MedlineCitation']"
            "/*[local-name()='PMID']/text()"
        )

        pmid = (
            pmid_values[0].strip()
            if pmid_values and pmid_values[0].strip()
            else None
        )

        title_nodes = article.xpath(
            ".//*[local-name()='ArticleTitle']"
        )

        title = (
            "".join(title_nodes[0].itertext()).strip()
            if title_nodes else None
        )

        journal_values = article.xpath(
            "./*[local-name()='MedlineCitation']"
            "/*[local-name()='Article']"
            "/*[local-name()='Journal']"
            "/*[local-name()='Title']/text()"
        )

        journal = (
            journal_values[0].strip()
            if journal_values else None
        )

        # Empreinte du contenu XML complet de l'article
        xml_content = etree.tostring(
            article,
            method="c14n"
        )

        content_hash = hashlib.sha256(
            xml_content
        ).hexdigest()

        records.append({
            "Fichier": file.name,
            "Numero_article": index,
            "PMID": pmid,
            "Titre": title,
            "Journal": journal,
            "Hash_XML": content_hash
        })

# ==========================================
# 3. Créer le dataset initial
# ==========================================

df_all = pd.DataFrame(records)

# ==========================================
# 4. Identifier les doublons strictement identiques
# ==========================================

duplicate_mask = df_all.duplicated(
    subset=["PMID", "Hash_XML"],
    keep="first"
)

df_identical_duplicates = df_all[duplicate_mask].copy()

# Conserver une seule occurrence de chaque article
df_cleaned = df_all[~duplicate_mask].copy()

# ==========================================
# 5. Identifier les PMID à examiner
# ==========================================

conflict_mask = df_cleaned.duplicated(
    subset=["PMID"],
    keep=False
)

df_to_review = df_cleaned[conflict_mask].copy()

# ==========================================
# 6. Ajouter un statut à chaque article
# ==========================================

df_cleaned["Statut"] = "Unique"

df_cleaned.loc[
    df_cleaned["PMID"].isin(df_to_review["PMID"]),
    "Statut"
] = "A verifier : meme PMID, contenu different"

# ==========================================
# 7. Sauvegarder les résultats
# ==========================================

df_cleaned.to_csv(
    CLEANED_DIR / "articles_cleaned_provisional.csv",
    index=False,
    encoding="utf-8-sig"
)

df_identical_duplicates.to_csv(
    REPORT_DIR / "articles_identical_duplicates.csv",
    index=False,
    encoding="utf-8-sig"
)

df_to_review.to_csv(
    REPORT_DIR / "articles_to_review.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================
# 8. Afficher le bilan
# ==========================================

print("========== BILAN DU NETTOYAGE ==========")

print("Articles initiaux :", len(df_all))

print(
    "Doublons strictement identiques supprimés :",
    len(df_identical_duplicates)
)

print("Articles conservés :", len(df_cleaned))

print(
    "PMID uniques dans le dataset :",
    df_cleaned["PMID"].nunique()
)

print(
    "Articles nécessitant une vérification :",
    len(df_to_review)
)

print("\n========== ARTICLES À VÉRIFIER ==========")
display(
    df_to_review[
        ["PMID", "Fichier", "Titre", "Journal"]
    ]
)

print("\nDataset provisoire enregistré dans :", CLEANED_DIR)


========== BILAN DU NETTOYAGE ==========
Articles initiaux : 3906
Doublons strictement identiques supprimés : 2
Articles conservés : 3904
PMID uniques dans le dataset : 3903
Articles nécessitant une vérification : 2

========== ARTICLES À VÉRIFIER ==========


,PMID,Fichier,Titre,Journal
852,26165617,1924.xml,Orbital Compartment Syndrome as the Presenting...,Urology
869,26165617,1941.xml,Orbital Compartment Syndrome as the Presenting...,Urology



Dataset provisoire enregistré dans : c:\Users\Ghada\Desktop\medical-data-warehouse\data\cleaned


In [38]:

from pathlib import Path
from lxml import etree
import difflib

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

files_to_compare = ["1924.xml", "1941.xml"]
target_pmid = "26165617"

def extract_article(file_name, target_pmid):

    file_path = PROCESSED_DIR / file_name
    tree = etree.parse(str(file_path))

    articles = tree.xpath(
        "//*[local-name()='PubmedArticle']"
    )

    for article in articles:

        pmids = article.xpath(
            "./*[local-name()='MedlineCitation']"
            "/*[local-name()='PMID']/text()"
        )

        if pmids and pmids[0].strip() == target_pmid:
            return article

    return None


# Extraire les deux versions
article_1 = extract_article(files_to_compare[0], target_pmid)
article_2 = extract_article(files_to_compare[1], target_pmid)

if article_1 is None or article_2 is None:
    print("Impossible de trouver l'article dans les deux fichiers.")

else:

    # Convertir chaque article en texte XML lisible
    xml_1 = etree.tostring(
        article_1,
        pretty_print=True,
        encoding="unicode"
    )

    xml_2 = etree.tostring(
        article_2,
        pretty_print=True,
        encoding="unicode"
    )

    # Comparer les deux contenus
    differences = list(
        difflib.unified_diff(
            xml_1.splitlines(),
            xml_2.splitlines(),
            fromfile=files_to_compare[0],
            tofile=files_to_compare[1],
            lineterm=""
        )
    )

    print("========== DIFFÉRENCES XML ==========")

    if differences:
        for line in differences:
            print(line)
    else:
        print("Aucune différence textuelle.")

    print("\nNombre de lignes différentes :",
          len(differences))


========== DIFFÉRENCES XML ==========
--- 1924.xml
+++ 1941.xml
@@ -194,259 +194,7 @@
             <ArticleId IdType="pubmed">26165617</ArticleId>
         </ArticleIdList>
     </PubmedData>
+</PubmedArticle>
 
 
 
-
-    <MedlineCitation Owner="NLM" Status="MEDLINE">
-        <PMID Version="1">26165588</PMID>
-        <DateCreated>
-            <Year>2015</Year>
-            <Month>10</Month>
-            <Day>19</Day>
-        </DateCreated>
-        <DateCompleted>
-            <Year>2016</Year>
-            <Month>02</Month>
-            <Day>01</Day>
-        </DateCompleted>
-        <Article PubModel="Print-Electronic">
-            <Journal>
-                <ISSN IssnType="Electronic">1527-3792</ISSN>
-                <JournalIssue CitedMedium="Internet">
-                    <Volume>194</Volume>
-                    <Issue>5</Issue>
-                    <PubDate>
-                        <Year>2015</Year>
-                        <Month>Nov</Month>
-                    </Pub

In [40]:

from pathlib import Path
from lxml import etree

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

for filename in ["1924.xml", "1941.xml"]:

    file_path = PROCESSED_DIR / filename
    tree = etree.parse(str(file_path))

    articles = tree.xpath(
        "//*[local-name()='PubmedArticle']"
    )

    print("\n==========", filename, "==========")
    print("Nombre de PubmedArticle :", len(articles))

    for i, article in enumerate(articles, start=1):

        # PMID directement associé à chaque MedlineCitation
        citations = article.xpath(
            "./*[local-name()='MedlineCitation']"
        )

        pmids = []

        for citation in citations:
            values = citation.xpath(
                "./*[local-name()='PMID']/text()"
            )
            pmids.append(
                values[0].strip() if values else "PMID absent"
            )

        print(f"Article XML {i} :")
        print("  Nombre de MedlineCitation :", len(citations))
        print("  PMID trouvés :", pmids)



========== 1924.xml ==========
Nombre de PubmedArticle : 1
Article XML 1 :
  Nombre de MedlineCitation : 2
  PMID trouvés : ['26165617', '26165588']

========== 1941.xml ==========
Nombre de PubmedArticle : 1
Article XML 1 :
  Nombre de MedlineCitation : 1
  PMID trouvés : ['26165617']


In [41]:

from pathlib import Path
from lxml import etree
import pandas as pd
import re

# ==========================================
# 1. CHEMINS DU PROJET
# ==========================================

PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CLEANED_DIR = PROJECT_DIR / "data" / "cleaned"
REPORT_DIR = PROJECT_DIR / "reports"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ==========================================
# 2. FONCTIONS D'EXTRACTION
# ==========================================

def get_text(node):
    """Extraire le texte d'un élément XML."""

    if node is None:
        return None

    value = " ".join(
        node.itertext()
    )

    value = re.sub(r"\s+", " ", value).strip()

    return value if value else None


def get_first_text(node, xpath):
    """Récupérer le premier texte correspondant au XPath."""

    results = node.xpath(xpath)

    if not results:
        return None

    if isinstance(results[0], str):
        return results[0].strip() or None

    return get_text(results[0])


def get_multiple_texts(node, xpath):
    """Récupérer plusieurs valeurs XML."""

    results = node.xpath(xpath)

    values = []

    for item in results:
        if isinstance(item, str):
            value = item.strip()
        else:
            value = get_text(item)

        if value and value not in values:
            values.append(value)

    return values


# ==========================================
# 3. EXTRACTION DES AUTEURS
# ==========================================

def extract_authors(citation):

    authors = []

    author_nodes = citation.xpath(
        "./*[local-name()='Article']"
        "/*[local-name()='AuthorList']"
        "/*[local-name()='Author']"
    )

    for author in author_nodes:

        collective = get_first_text(
            author,
            "./*[local-name()='CollectiveName']"
        )

        if collective:
            name = collective

        else:
            lastname = get_first_text(
                author,
                "./*[local-name()='LastName']"
            )

            forename = get_first_text(
                author,
                "./*[local-name()='ForeName']"
            )

            initials = get_first_text(
                author,
                "./*[local-name()='Initials']"
            )

            suffix = get_first_text(
                author,
                "./*[local-name()='Suffix']"
            )

            parts = [
                forename,
                lastname,
                suffix
            ]

            name = " ".join(
                part for part in parts if part
            )

            if not name:
                name = initials

        if name:
            authors.append(name)

    return authors


# ==========================================
# 4. EXTRACTION DES RÉSUMÉS
# ==========================================

def extract_abstract(citation):

    abstract_nodes = citation.xpath(
        "./*[local-name()='Article']"
        "/*[local-name()='Abstract']"
        "/*[local-name()='AbstractText']"
    )

    sections = []

    for section in abstract_nodes:

        text = get_text(section)

        if not text:
            continue

        label = section.get("Label")

        if label:
            sections.append(f"{label}: {text}")
        else:
            sections.append(text)

    return " ".join(sections) if sections else None


# ==========================================
# 5. EXTRACTION DES DATES
# ==========================================

def extract_publication_date(citation):

    pubdate_nodes = citation.xpath(
        "./*[local-name()='Article']"
        "/*[local-name()='Journal']"
        "/*[local-name()='JournalIssue']"
        "/*[local-name()='PubDate']"
    )

    if not pubdate_nodes:
        return {
            "Year": None,
            "Month": None,
            "Day": None,
            "Publication_Date": None
        }

    pubdate = pubdate_nodes[0]

    year = get_first_text(
        pubdate, "./*[local-name()='Year']"
    )

    month = get_first_text(
        pubdate, "./*[local-name()='Month']"
    )

    day = get_first_text(
        pubdate, "./*[local-name()='Day']"
    )

    medline_date = get_first_text(
        pubdate, "./*[local-name()='MedlineDate']"
    )

    publication_date = None

    if year:
        parts = [year]

        if month:
            parts.append(month)

        if day:
            parts.append(day)

        publication_date = "-".join(parts)

    elif medline_date:
        publication_date = medline_date

    return {
        "Year": year,
        "Month": month,
        "Day": day,
        "Publication_Date": publication_date
    }


# ==========================================
# 6. EXTRACTION DES DONNÉES D'UN ARTICLE
# ==========================================

def extract_article(citation, pubmed_data, filename,
                    article_index, citation_index,
                    citation_count):

    pmid = get_first_text(
        citation,
        "./*[local-name()='PMID']"
    )

    title = get_first_text(
        citation,
        "./*[local-name()='Article']"
        "/*[local-name()='ArticleTitle']"
    )

    journal = get_first_text(
        citation,
        "./*[local-name()='Article']"
        "/*[local-name()='Journal']"
        "/*[local-name()='Title']"
    )

    issn = get_first_text(
        citation,
        "./*[local-name()='Article']"
        "/*[local-name()='Journal']"
        "/*[local-name()='ISSN']"
    )

    authors = extract_authors(citation)

    abstract = extract_abstract(citation)

    date_info = extract_publication_date(citation)

    keywords = get_multiple_texts(
        citation,
        ".//*[local-name()='KeywordList']"
        "/*[local-name()='Keyword']"
    )

    affiliations = get_multiple_texts(
        citation,
        ".//*[local-name()='Affiliation']"
    )

    languages = get_multiple_texts(
        citation,
        "./*[local-name()='Article']"
        "/*[local-name()='Language']"
    )

    publication_types = get_multiple_texts(
        citation,
        ".//*[local-name()='PublicationTypeList']"
        "/*[local-name()='PublicationType']"
    )

    mesh_terms = get_multiple_texts(
        citation,
        ".//*[local-name()='MeshHeading']"
        "/*[local-name()='DescriptorName']"
    )

    dois = []

    if pubmed_data is not None:

        article_ids = pubmed_data.xpath(
            ".//*[local-name()='ArticleId']"
        )

        for article_id in article_ids:

            if article_id.get("IdType") == "doi":

                doi = get_text(article_id)

                if doi:
                    dois.append(doi)

    return {
        "Record_ID": f"{filename}_{article_index}_{citation_index}",
        "Source_File": filename,
        "PMID": pmid,
        "Title": title,
        "Authors": authors,
        "Authors_Count": len(authors),
        "Journal": journal,
        "ISSN": issn,
        "Publication_Date": date_info["Publication_Date"],
        "Year": date_info["Year"],
        "Month": date_info["Month"],
        "Day": date_info["Day"],
        "Abstract": abstract,
        "Keywords": keywords,
        "Affiliations": affiliations,
        "Language": languages,
        "DOI": list(dict.fromkeys(dois)),
        "Publication_Type": publication_types,
        "MeSH_Terms": mesh_terms,
        "Citation_Index": citation_index,
        "Citations_In_XML": citation_count,
        "Structural_Anomaly": citation_count > 1
    }


# ==========================================
# 7. TRAITER TOUS LES FICHIERS XML
# ==========================================

records = []
anomalies = []
errors = []

xml_files = sorted(PROCESSED_DIR.glob("*.xml"))

for i, file in enumerate(xml_files, start=1):

    print(f"[{i}/{len(xml_files)}] {file.name}")

    try:

        tree = etree.parse(str(file))

        articles = tree.xpath(
            "//*[local-name()='PubmedArticle']"
        )

        for article_index, article in enumerate(
            articles, start=1
        ):

            citations = article.xpath(
                "./*[local-name()='MedlineCitation']"
            )

            pubmed_data_nodes = article.xpath(
                "./*[local-name()='PubmedData']"
            )

            pubmed_data = (
                pubmed_data_nodes[0]
                if pubmed_data_nodes else None
            )

            if len(citations) > 1:

                anomalies.append({
                    "Fichier": file.name,
                    "Nombre_MedlineCitation": len(citations),
                    "Type": "Plusieurs MedlineCitation dans un PubmedArticle"
                })

            for citation_index, citation in enumerate(
                citations, start=1
            ):

                record = extract_article(
                    citation,
                    pubmed_data,
                    file.name,
                    article_index,
                    citation_index,
                    len(citations)
                )

                records.append(record)

    except Exception as e:

        errors.append({
            "Fichier": file.name,
            "Erreur": str(e)
        })


# ==========================================
# 8. CRÉER LE DATAFRAME
# ==========================================

df_bibliographic = pd.DataFrame(records)

df_anomalies = pd.DataFrame(anomalies)

df_errors = pd.DataFrame(errors)


# ==========================================
# 9. SAUVEGARDER LES DONNÉES
# ==========================================

df_bibliographic.to_csv(
    CLEANED_DIR / "bibliographic_extraction_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

df_anomalies.to_csv(
    REPORT_DIR / "bibliographic_structural_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)

df_errors.to_csv(
    REPORT_DIR / "bibliographic_extraction_errors.csv",
    index=False,
    encoding="utf-8-sig"
)


# ==========================================
# 10. AFFICHER LE BILAN
# ==========================================

print("\n========== BILAN EXTRACTION BIBLIOGRAPHIQUE ==========")

print("Fichiers XML traités :", len(xml_files))

print("Articles extraits :", len(df_bibliographic))

print("Articles sans PMID :",
      df_bibliographic["PMID"].isna().sum())

print("Articles sans titre :",
      df_bibliographic["Title"].isna().sum())

print("Articles sans résumé :",
      df_bibliographic["Abstract"].isna().sum())

print("Articles sans auteur :",
      df_bibliographic["Authors_Count"].eq(0).sum())

print("Articles sans date :",
      df_bibliographic["Publication_Date"].isna().sum())

print("Anomalies structurelles :", len(df_anomalies))

print("Erreurs d'extraction :", len(df_errors))

print("\nAperçu des données :")

display(
    df_bibliographic[
        [
            "PMID",
            "Title",
            "Authors",
            "Journal",
            "Publication_Date",
            "Language"
        ]
    ].head(10)
)

print("\nFichier enregistré dans :",
      CLEANED_DIR / "bibliographic_extraction_raw.csv")


[1/3905] 1000.xml
[2/3905] 1001.xml
[3/3905] 1002.xml
[4/3905] 1003.xml
[5/3905] 1004.xml
[6/3905] 1005.xml
[7/3905] 1006.xml
[8/3905] 1007.xml
[9/3905] 1008.xml
[10/3905] 1009.xml
[11/3905] 1010.xml
[12/3905] 1011.xml
[13/3905] 1012.xml
[14/3905] 1013.xml
[15/3905] 1014.xml
[16/3905] 1015.xml
[17/3905] 1016.xml
[18/3905] 1017.xml
[19/3905] 1018.xml
[20/3905] 1019.xml
[21/3905] 1020.xml
[22/3905] 1021.xml
[23/3905] 1022.xml
[24/3905] 1023.xml
[25/3905] 1024.xml
[26/3905] 1025.xml
[27/3905] 1026.xml
[28/3905] 1027.xml
[29/3905] 1028.xml
[30/3905] 1029.xml
[31/3905] 1030.xml
[32/3905] 1031.xml
[33/3905] 1032.xml
[34/3905] 1033.xml
[35/3905] 1034.xml
[36/3905] 1035.xml
[37/3905] 1036.xml
[38/3905] 1037.xml
[39/3905] 1038.xml
[40/3905] 1039.xml
[41/3905] 1040.xml
[42/3905] 1041.xml
[43/3905] 1042.xml
[44/3905] 1043.xml
[45/3905] 1044.xml
[46/3905] 1045.xml
[47/3905] 1046.xml
[48/3905] 1047.xml
[49/3905] 1048.xml
[50/3905] 1049.xml
[51/3905] 1050.xml
[52/3905] 1051.xml
[53/3905] 1052.xml
[5

,PMID,Title,Authors,Journal,Publication_Date,Language
0,26613881,Role of Multiparametric MR Imaging in Malignan...,"[Alberto Diaz de Leon, Daniel Costa, Ivan Pedr...",Magnetic resonance imaging clinics of North Am...,2016-Feb,[eng]
1,26600862,Evaluation of discoidin domain receptor-2 (DDR...,"[Mitra Azemikhah, Hamidreza Ahmadi Ashtiani, M...",Research in pharmaceutical sciences,2015 Jul-Aug,[eng]
2,26602027,Five-year biochemical recurrence-free and over...,"[Katharina Boehm, Jonas Schiffmann, Zhe Tian, ...",Urologic oncology,2016-Mar,[eng]
3,26602178,Cost-effectiveness of multiparametric magnetic...,"[Yannick Cerantola, Alice Dragomir, Simon Tang...",Urologic oncology,2016-Mar,[eng]
4,26600400,The small leucine rich proteoglycan fibromodul...,"[Niradiz Reyes, Ines Benedetti, Alfonso Bettin...",Cancer biomarkers : section A of Disease markers,2016-Jan-18,[eng]
5,26599623,A 3D in vitro model of patient-derived prostat...,"[Eliza L S Fong, Xinhai Wan, Jun Yang, Micaela...",Biomaterials,2016-Jan,[eng]
6,26599461,Disulfiram and its novel derivative sensitize ...,"[Vikas Sharma, Vikas Verma, Nand Lal, Santosh ...",Molecular carcinogenesis,2015-Nov-24,[ENG]
7,26599413,Cancer mortality in Itapúa-A rural province of...,"[Manuel Codas, Beate Pesch, Madita Adolphs, Ca...",Cancer epidemiology,2016-Feb,[eng]
8,26602092,Renal-cell carcinoma risk estimates based on p...,"[Yair Lotan, Jose A Karam, Shahrokh F Shariat,...",Urologic oncology,2016-Apr,[eng]
9,26595447,A novel method for identifying settings for we...,"[Andreas Stang, Bernd Kowall, Carsten Rusner, ...",International journal of cancer,2016-Apr-15,[eng]



Fichier enregistré dans : c:\Users\Ghada\Desktop\medical-data-warehouse\data\cleaned\bibliographic_extraction_raw.csv


In [44]:
# Bilan de qualité des données bibliographiques

import pandas as pd

# Charger les données extraites
df = pd.read_csv(
    PROJECT_DIR / "data" / "cleaned" / "bibliographic_extraction_raw.csv"
)

print("========== DIMENSIONS ==========")
print("Nombre de publications :", len(df))
print("Nombre de colonnes :", len(df.columns))

print("\n========== VALEURS MANQUANTES ==========")

missing = df.isnull().sum()

missing_report = pd.DataFrame({
    "Champ": missing.index,
    "Valeurs_manquantes": missing.values,
    "Pourcentage": (missing.values / len(df) * 100).round(2)
})

print(missing_report.to_string(index=False))

print("\n========== PMIDs DUPLIQUÉS ==========")
print("Nombre de PMIDs dupliqués :",
      df["PMID"].duplicated().sum())

print("\n========== ANOMALIES STRUCTURELLES ==========")
if "Structural_Anomaly" in df.columns:
    print(df["Structural_Anomaly"].value_counts(dropna=False))

print("\n========== LANGUES ==========")
print(df["Language"].value_counts(dropna=False).head(20))

print("\n========== APERÇU DES DATES ==========")
print(df["Publication_Date"].value_counts(dropna=False).head(20))

========== DIMENSIONS ==========
Nombre de publications : 3907
Nombre de colonnes : 22

========== VALEURS MANQUANTES ==========
             Champ  Valeurs_manquantes  Pourcentage
         Record_ID                   0         0.00
       Source_File                   0         0.00
              PMID                   0         0.00
             Title                   0         0.00
           Authors                   0         0.00
     Authors_Count                   0         0.00
           Journal                   0         0.00
              ISSN                   6         0.15
  Publication_Date                   0         0.00
              Year                 170         4.35
             Month                 953        24.39
               Day                3181        81.42
          Abstract                  11         0.28
          Keywords                   0         0.00
      Affiliations                   0         0.00
          Language                   0 

In [45]:
# Diagnostic des doublons et des dates

# 1. Vérification des PMIDs dupliqués
duplicates = df[df["PMID"].duplicated(keep=False)].sort_values("PMID")

print("========== PMIDs DUPLIQUÉS ==========")
print("Nombre de lignes concernées :", len(duplicates))

if not duplicates.empty:
    print(duplicates[
        ["PMID", "Title", "Source_File", "Publication_Date", "Year"]
    ].to_string(index=False))
else:
    print("Aucun PMID dupliqué.")

# 2. Publications sans année extraite
missing_year = df[df["Year"].isna()]

print("\n========== PUBLICATIONS SANS ANNÉE ==========")
print("Nombre :", len(missing_year))

print(
    missing_year[
        ["PMID", "Title", "Publication_Date", "Source_File"]
    ].head(30).to_string(index=False)
)

# 3. Vérifier les formats de dates
print("\n========== FORMATS DE DATES ==========")
print(df["Publication_Date"].value_counts().head(30).to_string())

# 4. Vérifier les langues
print("\n========== LANGUES AVANT NORMALISATION ==========")
print(df["Language"].value_counts(dropna=False).to_string())

========== PMIDs DUPLIQUÉS ==========
Nombre de lignes concernées : 6
    PMID                                                                                                                                                                                 Title Source_File Publication_Date   Year
26165617                                          Orbital Compartment Syndrome as the Presenting Sign of Disseminated Intravascular Coagulation From Metastatic Prostatic Cancer to the Orbit.    1924.xml         2015-Aug 2015.0
26165617                                          Orbital Compartment Syndrome as the Presenting Sign of Disseminated Intravascular Coagulation From Metastatic Prostatic Cancer to the Orbit.    1941.xml         2015-Aug 2015.0
26560494                                                                                                                                 Prospects for the future of prostate cancer vaccines.    1116.xml         2016-Mar 2016.0
26560494              

nettoyage 

In [46]:
import re
import ast
import json
import pandas as pd
from pathlib import Path

# Charger les données brutes extraites
input_file = PROJECT_DIR / "data" / "cleaned" / "bibliographic_extraction_raw.csv"
df = pd.read_csv(input_file)

df_clean = df.copy()

# ---------------------------------------
# 1. Nettoyage des champs textuels
# ---------------------------------------

text_columns = [
    "Title", "Journal", "Publication_Date", "DOI",
    "ISSN", "Abstract"
]

for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

        # Remplacer les chaînes vides par des valeurs manquantes
        df_clean[col] = df_clean[col].replace("", pd.NA)


# ---------------------------------------
# 2. Normalisation des langues
# ---------------------------------------

def normalize_language(value):
    if pd.isna(value):
        return pd.NA

    try:
        languages = ast.literal_eval(str(value))
        if isinstance(languages, list):
            languages = [str(lang).strip().lower() for lang in languages]
            return json.dumps(languages, ensure_ascii=False)
    except (ValueError, SyntaxError):
        pass

    return str(value).strip().lower()


df_clean["Language"] = df_clean["Language"].apply(normalize_language)


# ---------------------------------------
# 3. Récupération des années manquantes
# ---------------------------------------

# Conserver les années déjà extraites
df_clean["Year"] = pd.to_numeric(
    df_clean["Year"], errors="coerce"
).astype("Int64")

def extract_year(date):
    if pd.isna(date):
        return pd.NA

    match = re.search(r"\b(?:19|20)\d{2}\b", str(date))

    if match:
        return int(match.group())

    return pd.NA


for index, row in df_clean.iterrows():
    if pd.isna(row["Year"]):
        extracted_year = extract_year(row["Publication_Date"])

        if not pd.isna(extracted_year):
            df_clean.at[index, "Year"] = extracted_year


# ---------------------------------------
# 4. Vérification des doublons
# ---------------------------------------

# Sauvegarder les doublons avant toute suppression
duplicates_report = df_clean[
    df_clean.duplicated(subset=["PMID"], keep=False)
].sort_values("PMID")

duplicates_report.to_csv(
    PROJECT_DIR / "reports" / "bibliographic_duplicates_before_cleaning.csv",
    index=False,
    encoding="utf-8-sig"
)

# Préférer une ligne sans anomalie structurelle lorsqu'elle existe
if "Structural_Anomaly" in df_clean.columns:
    df_clean["_anomaly_sort"] = (
        df_clean["Structural_Anomaly"]
        .fillna(False)
        .astype(bool)
    )
    df_clean = df_clean.sort_values(
        by=["_anomaly_sort", "Source_File"]
    )

# Conserver une seule ligne par PMID
df_clean = df_clean.drop_duplicates(
    subset=["PMID"],
    keep="first"
)

df_clean = df_clean.drop(columns=["_anomaly_sort"], errors="ignore")


# ---------------------------------------
# 5. Sauvegarde du fichier nettoyé
# ---------------------------------------

output_file = PROJECT_DIR / "data" / "cleaned" / "bibliographic_cleaned.csv"

df_clean.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


# ---------------------------------------
# 6. Bilan après nettoyage
# ---------------------------------------

print("========== BILAN DU NETTOYAGE ==========")

print("Publications avant nettoyage :", len(df))
print("Publications après déduplication :", len(df_clean))
print("Doublons supprimés :", len(df) - len(df_clean))

print("\nAnnées manquantes avant :", df["Year"].isna().sum())
print("Années manquantes après :", df_clean["Year"].isna().sum())

print("\nLangues après normalisation :")
print(df_clean["Language"].value_counts().head(15))

print("\nPMID encore dupliqués :",
      df_clean["PMID"].duplicated().sum())

print("\nFichier sauvegardé :", output_file)

========== BILAN DU NETTOYAGE ==========
Publications avant nettoyage : 3907
Publications après déduplication : 3904
Doublons supprimés : 3

Années manquantes avant : 170
Années manquantes après : 0

Langues après normalisation :
Language
["eng"]           3713
["fre"]             44
["chi"]             33
["ger"]             28
["eng", "spa"]      25
["jpn"]             17
["spa"]             17
["rus"]             10
["dan"]              4
["hun"]              3
["dut"]              2
["eng", "fre"]       2
["pol"]              2
["ita"]              1
["cze"]              1
Name: count, dtype: int64

PMID encore dupliqués : 0

Fichier sauvegardé : c:\Users\Ghada\Desktop\medical-data-warehouse\data\cleaned\bibliographic_cleaned.csv


In [47]:
# Vérification des publications concernées par les anomalies

pmids_to_check = ["26165617", "26165588", "26560494", "26568444"]

verification = df_clean[
    df_clean["PMID"].astype(str).isin(pmids_to_check)
][
    ["PMID", "Title", "Source_File", "Year"]
]

print("========== VÉRIFICATION DES PMID ==========")
print(verification.to_string(index=False))

print("\n========== CONTRÔLE GLOBAL ==========")
print("Nombre de publications :", len(df_clean))
print("Nombre de PMID uniques :", df_clean["PMID"].nunique())
print("PMID 26165588 présent :",
      (df_clean["PMID"].astype(str) == "26165588").any())

========== VÉRIFICATION DES PMID ==========
    PMID                                                                                                                                                                                 Title Source_File  Year
26568444 Which patients benefit from post-implant CT dosimetry after real-time intraoperative planning for low dose rate prostate brachytherapy? Case series and systematic literature review.    1095.xml  2016
26560494                                                                                                                                 Prospects for the future of prostate cancer vaccines.    1116.xml  2016
26165617                                          Orbital Compartment Syndrome as the Presenting Sign of Disseminated Intravascular Coagulation From Metastatic Prostatic Cancer to the Orbit.    1941.xml  2015
26165588                                                                Baseline Prostate Atrophy is Associated with Red

In [48]:
import pandas as pd

df_clean = pd.read_csv(
    PROJECT_DIR / "data" / "cleaned" / "bibliographic_cleaned.csv"
)

print("========== COLONNES DISPONIBLES ==========")
print(df_clean.columns.tolist())

print("\n========== EXEMPLE D'UNE PUBLICATION ==========")

for column in df_clean.columns:
    print(f"\n{column} :")
    print(df_clean[column].iloc[0])

print("\n========== TYPES DE DONNÉES ==========")
print(df_clean.dtypes)

========== COLONNES DISPONIBLES ==========
['Record_ID', 'Source_File', 'PMID', 'Title', 'Authors', 'Authors_Count', 'Journal', 'ISSN', 'Publication_Date', 'Year', 'Month', 'Day', 'Abstract', 'Keywords', 'Affiliations', 'Language', 'DOI', 'Publication_Type', 'MeSH_Terms', 'Citation_Index', 'Citations_In_XML', 'Structural_Anomaly']

========== EXEMPLE D'UNE PUBLICATION ==========

Record_ID :
1000.xml_1_1

Source_File :
1000.xml

PMID :
26613881

Title :
Role of Multiparametric MR Imaging in Malignancies of the Urogenital Tract.

Authors :
['Alberto Diaz de Leon', 'Daniel Costa', 'Ivan Pedrosa']

Authors_Count :
3

Journal :
Magnetic resonance imaging clinics of North America

ISSN :
1557-9786

Publication_Date :
2016-Feb

Year :
2016

Month :
Feb

Day :
nan

Abstract :
Multiparametric MR imaging (mpMRI) combine different sequences that, properly tailored, can provide qualitative and quantitative information about the tumor microenvironment beyond traditional tumor size measures and/or 

In [49]:
# Inspection des champs multivalués

columns_to_check = [
    "Authors",
    "Keywords",
    "Affiliations",
    "Language",
    "Publication_Type",
    "MeSH_Terms"
]

for col in columns_to_check:
    print("\n" + "=" * 50)
    print("COLONNE :", col)

    print("Type :", df_clean[col].dtype)

    print("\nExemples :")
    for value in df_clean[col].dropna().head(3):
        print(value)

    print("\nNombre de valeurs manquantes :",
          df_clean[col].isna().sum())


COLONNE : Authors
Type : object

Exemples :
['Alberto Diaz de Leon', 'Daniel Costa', 'Ivan Pedrosa']
['Mitra Azemikhah', 'Hamidreza Ahmadi Ashtiani', 'Mahmoud Aghaei', 'Hosein Rastegar']
['Katharina Boehm', 'Jonas Schiffmann', 'Zhe Tian', 'Hans Lesmana', 'Alessandro Larcher', 'Philipp Mandel', 'Pierre I Karakiewicz', 'Markus Graefen', 'Rudolf Schwarz', 'Andreas Krüll', 'Derya Tilki']

Nombre de valeurs manquantes : 0

COLONNE : Keywords
Type : object

Exemples :
['Arterial spin labeling', 'Diffusion-weighted imaging', 'Dynamic contrast-enhanced MR imaging', 'Kidney cancer', 'MR imaging', 'Prostate cancer', 'Urothelial carcinoma']
['Discoidin domain receptor2 (DDR2)', 'Prognostic factors', 'Prostate cancer']
['Biochemical recurrence', 'Brachytherapy', 'High dose rate', 'Prostate cancer', 'Radical prostatectomy']

Nombre de valeurs manquantes : 0

COLONNE : Affiliations
Type : object

Exemples :
['Department of Radiology, University of Texas Southwestern Medical Center, 2201 Inwood Road